<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session13_building_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۳: ساخت ایجنت از صفر

### اهداف این جلسه

- تفاوت گردش کار (Workflow) و ایجنت (Agent) و اینکه کدام کی مناسب است
- پیاده‌سازی حلقهٔ ReAct: فکر، اقدام، مشاهده
- ساخت ایجنتی که RAG، کاتالوگ کالا و ماشین‌حساب را ترکیب می‌کند
- مدیریت خطاهای ایجنت: حلقهٔ تکراری، سقف گام و ردیابی (Trace)
- ارزیابی ایجنت با مجموعه‌ای از کارها: نرخ موفقیت، تعداد گام و زمان

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. گردش کار یا ایجنت؟

در جلسهٔ قبل حلقهٔ ابزار را ساختیم. یک **ایجنت (Agent)** همان حلقه است با یک تفاوت مهم در نگاه: مدل خودش تصمیم می‌گیرد **چه گام‌هایی، به چه ترتیبی و تا کی** برداشته شوند.

| | گردش کار (Workflow) | ایجنت (Agent) |
|---|---|---|
| مسیر اجرا | از پیش تعیین‌شده در کد | در زمان اجرا توسط مدل |
| پیش‌بینی‌پذیری | بالا | پایین‌تر |
| هزینه و تأخیر | کم و ثابت | متغیر، معمولاً بیشتر |
| مناسب برای | کارهای تکراری با مراحل معلوم | کارهای باز با مراحل نامعلوم |
| نمونه | RAG جلسهٔ ۸: بازیابی، سپس پاسخ | «با این بودجه چه گوشی‌ای بخرم و قسطی چقدر می‌شود؟» |

قاعدهٔ عملی: **با ساده‌ترین راه‌حل شروع کنید.** اگر مراحل کار را از پیش می‌دانید، گردش کار بنویسید. ایجنت را برای جایی نگه دارید که واقعاً نمی‌دانید چند گام و کدام ابزارها لازم است.

## ۲. حلقهٔ ReAct

الگوی **ReAct** (Reasoning + Acting، Yao و همکاران، ۲۰۲۲) سه جزء دارد که تکرار می‌شوند:

1. **فکر (Thought):** مدل می‌گوید چه چیزی را باید بفهمد.
2. **اقدام (Action):** یک ابزار را صدا می‌زند.
3. **مشاهده (Observation):** نتیجهٔ ابزار را می‌بیند.

این چرخه تا رسیدن به پاسخ نهایی ادامه پیدا می‌کند. در مقالهٔ اصلی، فکر و اقدام به شکل متن آزاد نوشته و با عبارت‌های منظم تجزیه می‌شدند. امروز با Tool Calling بومی، اقدام ساختاریافته است و فکر در متن همراه فراخوانی ابزار می‌آید.

</div>

In [ ]:
!pip -q install openai sentence-transformers hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. ابزارهای ایجنت

ایجنت ما به همهٔ منابع فروشگاه دسترسی دارد: اسناد سیاست‌ها از طریق RAG جلسهٔ ۸، به شکل یک ابزار، و ابزارهای جلسهٔ ۱۲. **RAG به‌عنوان ابزار** یعنی ایجنت خودش تصمیم می‌گیرد کی و با چه عبارتی در اسناد جستجو کند، و اگر نتیجهٔ اول کافی نبود، دوباره جستجو کند.

</div>

In [ ]:
import json

import numpy as np
from hazm import Normalizer
from llm_utils import download_data
from sentence_transformers import SentenceTransformer
from store_tools import STORE_TOOLS, run_tool, tool_schema

chunks = [json.loads(line) for line in open(download_data("novinkala_chunks.jsonl"), encoding="utf-8")]
_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
embedder = SentenceTransformer("intfloat/multilingual-e5-base")
chunk_vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)


def search_docs(query: str) -> list:
    """Search the Novin Kala policy documents (shipping, returns, warranty, payment, instalments,
    club, orders, account, installation, support, gift cards, corporate sales).

    Args:
        query: a short question or keywords in Persian, e.g. "شرایط خرید اقساطی".
    """
    q = embedder.encode([f"query: {_normalizer.normalize(query).translate(_DIGITS)}"], normalize_embeddings=True)[0]
    top = np.argsort(-(chunk_vectors @ q))[:3]
    return [{"source": chunks[i]["id"], "text": chunks[i]["text"]} for i in top]


TOOLS = [search_docs] + STORE_TOOLS
for fn in TOOLS:
    print(f"{fn.__name__:16} {tool_schema(fn)['function']['description'][:80]}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. کلاس Agent

کلاس زیر حلقهٔ ReAct را با سه سازوکار ایمنی پیاده می‌کند:

- **سقف گام (`max_steps`):** ایجنت هرگز بی‌نهایت اجرا نمی‌شود.
- **تشخیص تکرار:** اگر ایجنت دقیقاً همان فراخوانی را دوباره بخواهد، به جای اجرا به او گفته می‌شود که در حال دور زدن است.
- **ردیابی (Trace):** همهٔ گام‌ها ثبت می‌شوند تا بتوانیم رفتار ایجنت را بررسی و ارزیابی کنیم.

</div>

In [ ]:
import time

AGENT_SYSTEM = """You are the shopping assistant of Novin Kala, an Iranian online store. Answer in Persian.
You can use tools. Work step by step:
- Store rules (shipping, returns, warranty, payment, instalments, ...) come ONLY from search_docs.
- Products, prices and stock come ONLY from search_products or get_product.
- Do every calculation with the calculator tool.
- Before each tool call, write one short sentence about what you are going to check.
- When you have everything you need, give a short, clear final answer with the numbers you used.
- If the information is not available, say so; never invent prices, rules or numbers."""


class Agent:
    def __init__(self, tools, system=AGENT_SYSTEM, max_steps=8):
        self.tools, self.system, self.max_steps = tools, system, max_steps
        self.schemas = [tool_schema(t) for t in tools]

    def run(self, task, verbose=True):
        messages = [{"role": "system", "content": self.system}, {"role": "user", "content": task}]
        trace, seen, start = [], set(), time.time()
        for step in range(1, self.max_steps + 1):
            message = client.chat.completions.create(model=MODEL, messages=messages, tools=self.schemas,
                                                     temperature=0).choices[0].message
            if message.content and verbose:
                print(f"[{step}] thought: {message.content.strip()[:200]}")
            if not message.tool_calls:
                trace.append({"step": step, "type": "answer"})
                return {"answer": message.content, "trace": trace, "steps": step,
                        "seconds": time.time() - start}
            messages.append(message.model_dump(exclude_none=True))
            for call in message.tool_calls:
                key = (call.function.name, call.function.arguments)
                if key in seen:                       # the same call again: the agent is going in circles
                    result = json.dumps({"error": "you already made exactly this call; use its result "
                                                  "or try something different"})
                else:
                    result = run_tool(call.function.name, call.function.arguments, self.tools)
                seen.add(key)
                trace.append({"step": step, "type": "tool", "name": call.function.name,
                              "arguments": call.function.arguments, "result": result[:300]})
                if verbose:
                    print(f"[{step}] action: {call.function.name}({call.function.arguments})")
                    print(f"     observation: {result[:160]}")
                messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
        return {"answer": "متأسفانه نتوانستم در تعداد گام مجاز به پاسخ برسم.", "trace": trace,
                "steps": self.max_steps, "seconds": time.time() - start}


agent = Agent(TOOLS)

In [ ]:
result = agent.run("می‌خوام یه گلکسی A55 قسطی بخرم. پیش‌پرداختش چقدر میشه؟")
print("\nFINAL:", result["answer"])
print(f"steps: {result['steps']}   time: {result['seconds']:.1f}s")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای جواب دادن به این پرسش، ایجنت باید سه منبع را ترکیب کند: قاعدهٔ خرید اقساطی از اسناد، یعنی پیش‌پرداخت ۲۵ درصد و حداقل مبلغ ۵ میلیون تومان، قیمت کالا از کاتالوگ، و محاسبه. پاسخ درست: $28{,}900{,}000 \times 0.25 = 7{,}225{,}000$ تومان.

Trace را بخوانید: آیا ایجنت قاعده را از اسناد گرفت یا از حافظهٔ خودش؟ حساب را با ماشین‌حساب انجام داد؟ ایجنتی که پاسخ درست را از راه غلط پیدا کرده، روی پرسش بعدی شکست می‌خورد.

</div>

In [ ]:
for task in ["ارزون‌ترین تلویزیون موجودتون چیه؟ اگه بفرستید شیراز با ارسال عادی، کل هزینه چقدر میشه؟",
             "سفارش NK519027 من کی می‌رسه؟ اگه هنوز ارسال نشده، می‌تونم لغوش کنم؟"]:
    print("=" * 80, f"\nTASK: {task}")
    result = agent.run(task)
    print("\nFINAL:", result["answer"])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. خطاهای رایج ایجنت

| خطا | نشانه در Trace | راه مقابله |
|---|---|---|
| حلقهٔ تکراری | یک فراخوانی چند بار با همان آرگومان | تشخیص تکرار، سقف گام |
| دانش از حافظه | پاسخ قاعده‌ای که `search_docs` صدا زده نشده | قاعدهٔ صریح در پرامپت، ارزیابی منبع |
| حساب ذهنی | عدد نهایی بدون فراخوانی ماشین‌حساب | قاعدهٔ صریح، بررسی در ارزیابی |
| آرگومان اشتباه | خطای اعتبارسنجی | پیام خطای روشن؛ ایجنت خودش اصلاح می‌کند |
| توقف زودهنگام | پاسخ ناقص بعد از یک ابزار | بهبود پرامپت؛ مدل قوی‌تر |
| ابزار خیالی | نام ابزاری که وجود ندارد | `run_tool` خطا برمی‌گرداند |

بیشتر این خطاها با **خواندن Trace** پیدا می‌شوند، نه با خواندن پاسخ نهایی.

## ۶. ارزیابی ایجنت

ایجنت را نمی‌توان با یکی دو پرسش سنجید. مجموعه‌ای از **کارها (Tasks)** می‌سازیم که هر کدام یک بررسی‌کنندهٔ خودکار دارد: عددی که باید در پاسخ باشد و ابزارهایی که باید استفاده شوند. این همان روش جلسهٔ ۵ است، برای رفتار چندمرحله‌ای.

</div>

In [ ]:
import re

import pandas as pd

TASKS = [
    {"task": "پیش‌پرداخت خرید اقساطی یک گلکسی A55 چقدر است؟",
     "expect_number": 7_225_000, "expect_tools": {"search_docs", "calculator"}},
    {"task": "هزینه ارسال عادی برای سفارش ۸۰۰ هزار تومانی به آمل چقدر است؟",
     "expect_number": 45_000, "expect_tools": {"shipping_cost"}},
    {"task": "قیمت دو تا هدفون سونی XM5 با ارسال سریع به تهران جمعاً چقدر می‌شود؟",
     "expect_number": 43_885_000, "expect_tools": {"search_products", "shipping_cost"}},
    {"task": "گارانتی لپ‌تاپ IdeaPad چند ماه است؟",
     "expect_number": 18, "expect_tools": {"search_docs"}},
    {"task": "وضعیت سفارش NK482913 چیست؟",
     "expect_number": None, "expect_tools": {"track_order"}},
    {"task": "اگر ۳۰۰ امتیاز باشگاه داشته باشم، به چقدر اعتبار کیف پول تبدیل می‌شود؟",
     "expect_number": 1_500_000, "expect_tools": {"search_docs"}},
]


def numbers_in(text):
    text = (text or "").translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789"))
    text = re.sub(r"(?<=\d)[,٬](?=\d{3})", "", text)          # 43,885,000 -> 43885000
    return {int(float(n)) for n in re.findall(r"\d+(?:\.\d+)?", text)}


def evaluate_agent(agent, tasks=TASKS):
    rows = []
    for t in tasks:
        r = agent.run(t["task"], verbose=False)
        used = {s["name"] for s in r["trace"] if s["type"] == "tool"}
        number_ok = t["expect_number"] is None or t["expect_number"] in numbers_in(r["answer"])
        rows.append({"task": t["task"][:45], "success": number_ok, "tools_ok": t["expect_tools"] <= used,
                     "steps": r["steps"], "tool_calls": sum(s["type"] == "tool" for s in r["trace"]),
                     "seconds": round(r["seconds"], 1)})
    return pd.DataFrame(rows)


report = evaluate_agent(agent)
print(f"success: {report.success.mean():.0%}   expected tools used: {report.tools_ok.mean():.0%}   "
      f"mean steps: {report.steps.mean():.1f}")
report

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

دو معیار جداگانه داریم: **موفقیت**، یعنی عدد درست در پاسخ، و **مسیر درست**، یعنی استفاده از ابزارهای لازم. ایجنتی که عدد درست را بدون ابزار و از حافظه گفته، این بار شانس آورده است. برای ایجنت‌ها هر دو را بسنجید.

چون ایجنت‌ها تصادفی‌اند، حتی با دمای صفر، چون مسیر می‌تواند به تفاوت‌های کوچک حساس باشد، ارزیابی را چند بار اجرا کنید. نرخ موفقیت یک‌باره روی ۶ کار، عدم قطعیت بسیار زیادی دارد.

## ۷. همیشه ایجنت لازم نیست

برای پرسش «پیش‌پرداخت اقساطی کالای X چقدر است؟» مراحل کاملاً معلوم‌اند: قیمت کالا را پیدا کن و در ۰٫۲۵ ضرب کن. یک گردش کار ساده:

</div>

In [ ]:
from store_tools import search_products


def installment_downpayment(product_query):
    """A fixed workflow: no model decisions, the rule comes from the policy document."""
    product = search_products(product_query)[0]
    if product["price_toman"] <= 5_000_000:
        return f"{product['name']}: خرید اقساطی فقط برای سبد بیش از ۵ میلیون تومان ممکن است."
    return f"{product['name']}: پیش‌پرداخت ۲۵٪ = {int(product['price_toman'] * 0.25):,} تومان"


print(installment_downpayment("گلکسی A55"))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این تابع سریع، ارزان و همیشه درست است، ولی فقط همین یک نوع پرسش را جواب می‌دهد. در محصول واقعی معمولاً ترکیبی داریم: پرسش‌های رایج و مهم با گردش کار ثابت جواب داده می‌شوند و ایجنت برای پرسش‌های باز و غیرمنتظره است. **مسیریابی (Routing)** بین این دو را در جلسهٔ ۱۶ می‌سازیم.

## ۸. جمع‌بندی

- ایجنت حلقه‌ای است که در آن مدل گام‌ها را خودش انتخاب می‌کند. گردش کار مسیر ثابتی دارد. با ساده‌ترین راه شروع کنید.
- ReAct: فکر، اقدام، مشاهده. با Tool Calling بومی، اقدام ساختاریافته است.
- RAG به‌عنوان ابزار به ایجنت اجازه می‌دهد خودش تصمیم بگیرد کی و چطور در اسناد جستجو کند.
- سقف گام، تشخیص تکرار و Trace، حداقل سازوکارهای ایمنی و اشکال‌زدایی‌اند.
- ایجنت را با مجموعه‌ای از کارها ارزیابی کنید: هم نتیجه و هم مسیر.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. به ایجنت یک گام **برنامه‌ریزی** اضافه کنید: پیش از شروع حلقه، از مدل بخواهید فهرست گام‌ها را بنویسد و این برنامه را به پیام کاربر اضافه کنید. نرخ موفقیت و تعداد گام‌ها را با نسخهٔ بدون برنامه مقایسه کنید.
۲. تشخیص تکرار فعلی فقط فراخوانی **کاملاً یکسان** را می‌گیرد. آن را طوری گسترش دهید که اگر ایجنت سه بار پشت سر هم یک ابزار را صدا زد، حتی با آرگومان‌های متفاوت، هشدار بگیرد. کاری طراحی کنید که این حالت را نشان دهد.
۳. چهار کار جدید به `TASKS` اضافه کنید که دست‌کم دو تای آن‌ها پاسخی در اسناد ندارند و ایجنت باید بگوید اطلاعات موجود نیست. بررسی‌کنندهٔ مناسب برای این کارها بنویسید.
۴. گردش کار ثابتی برای پرسش «هزینهٔ کل خرید n عدد از کالای X با روش ارسال Y به شهر Z» بنویسید. روی ۱۰ نمونه، دقت و زمان آن را با ایجنت مقایسه کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** به ایجنت یک گام **برنامه‌ریزی** اضافه کنید: پیش از شروع حلقه، از مدل بخواهید فهرست گام‌ها را بنویسد و این برنامه را به پیام کاربر اضافه کنید. نرخ موفقیت و تعداد گام‌ها را با نسخهٔ بدون برنامه مقایسه کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** تشخیص تکرار فعلی فقط فراخوانی **کاملاً یکسان** را می‌گیرد. آن را طوری گسترش دهید که اگر ایجنت سه بار پشت سر هم یک ابزار را صدا زد، حتی با آرگومان‌های متفاوت، هشدار بگیرد. کاری طراحی کنید که این حالت را نشان دهد.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** چهار کار جدید به `TASKS` اضافه کنید که دست‌کم دو تای آن‌ها پاسخی در اسناد ندارند و ایجنت باید بگوید اطلاعات موجود نیست. بررسی‌کنندهٔ مناسب برای این کارها بنویسید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** گردش کار ثابتی برای پرسش «هزینهٔ کل خرید n عدد از کالای X با روش ارسال Y به شهر Z» بنویسید. روی ۱۰ نمونه، دقت و زمان آن را با ایجنت مقایسه کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید